In [ ]:
import os
import gc
import logging
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

from obj_model import SpatioTemporalDynamicClassifier
from dataset_factory import ContrastiveLearningDataset


def setup_logging(log_dir="logs_objectfolder"):
    os.makedirs(log_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = os.path.join(log_dir, f"train_objectfolder_{timestamp}.log")

    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(message)s',
        handlers=[
            logging.StreamHandler(),
            logging.FileHandler(log_file, encoding='utf-8')
        ]
    )
    logger = logging.getLogger(__name__)
    logger.info(f"日志保存至: {log_file}")
    return logger


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def accuracy(output, target, topk=(1,)):
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)
        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))
        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res


def load_mambavision_pretrained(model, ckpt_path, verbose=True):
    if not os.path.exists(ckpt_path):
        print(f"⚠️ 找不到预训练权重: {ckpt_path}，使用随机初始化")
        return []

    ckpt = torch.load(ckpt_path, map_location="cpu")
    state = ckpt.get("state_dict", ckpt.get("model", ckpt))
    model_state = model.state_dict()
    loaded = []

    for k, v in state.items():
        if "head" in k or "classifier" in k:
            continue
        target_key_v = f"vision_encoder.model.{k}"
        target_key_t = f"tactile_encoder.model.{k}"

        if target_key_v in model_state and model_state[target_key_v].shape == v.shape:
            model_state[target_key_v].copy_(v)
            loaded.append(target_key_v)
        if target_key_t in model_state and model_state[target_key_t].shape == v.shape:
            model_state[target_key_t].copy_(v)
            loaded.append(target_key_t)

    if verbose:
        print(f"[Pretrain] 成功加载 {len(loaded)} 个参数")
    return loaded


def build_optimizer(model, lr_backbone, lr_head, weight_decay):
    backbone_decay, backbone_no_decay = [], []
    head_decay, head_no_decay = [], []

    def is_backbone(name):
        return (name.startswith("vision_encoder.") or
                name.startswith("tactile_encoder.") or
                name.startswith("v_spatial_proj.") or
                name.startswith("t_proj."))

    def is_no_decay(name, p):
        if name.endswith(".bias") or p.ndim <= 1:
            return True
        if "norm" in name.lower() or "pos" in name.lower():
            return True
        return False

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if is_backbone(name):
            (backbone_no_decay if is_no_decay(name, p) else backbone_decay).append(p)
        else:
            (head_no_decay if is_no_decay(name, p) else head_decay).append(p)

    param_groups = [
        {"params": backbone_decay,    "lr": lr_backbone, "weight_decay": weight_decay},
        {"params": backbone_no_decay, "lr": lr_backbone, "weight_decay": 0.0},
        {"params": head_decay,        "lr": lr_head,     "weight_decay": weight_decay},
        {"params": head_no_decay,     "lr": lr_head,     "weight_decay": 0.0},
    ]
    return torch.optim.AdamW(
        [g for g in param_groups if len(g["params"]) > 0],
        betas=(0.9, 0.98), eps=1e-6
    )


@torch.no_grad()
def evaluate(model, loader, device, criterion, use_amp=True):
    model.eval()
    total_loss, total_correct, total_count = 0.0, 0, 0

    for batch in loader:


        rgb = batch[0].to(device, non_blocking=True).unsqueeze(1)
        tac = batch[2].to(device, non_blocking=True).unsqueeze(1)
        label = batch[4].to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):
            logits = model(rgb, tac)
            loss = criterion(logits, label)

        preds = logits.argmax(dim=1)
        total_loss += loss.item() * label.size(0)
        total_correct += (preds == label).sum().item()
        total_count += label.size(0)

    avg_loss = total_loss / max(1, total_count)
    avg_acc = 100.0 * total_correct / max(1, total_count)
    return avg_loss, avg_acc


def main():
    logger = setup_logging()


    config = {

        "data_folder": "",
        "split_file":  "",


        "train_dataset_name": "objectfolder_train",
        "test_dataset_name":  "objectfolder_test",
        "num_classes": 7,


        "epochs":       50,
        "batch_size":   32,
        "num_workers":  4,
        "grad_clip":    1.0,
        "lr_backbone":  3e-6,
        "lr_head":      3e-5,
        "weight_decay": 1e-2,
        "label_smoothing": 0.1,
        "dropout":      0.3,


        "d_model":      256,
        "d_state":      16,
        "hierarchical": True,


        "pretrained_path": "mambavision_tiny_1k.pth.tar",


        "save_dir": "checkpoints_objectfolder",
    }

    os.makedirs(config["save_dir"], exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    logger.info("=" * 60)
    logger.info("ObjectFolder Real 泛化实验 (单帧输入 T=1)")
    logger.info(f"任务: {config['train_dataset_name']} | 类别数: {config['num_classes']}")
    logger.info(f"设备: {device}")
    for k, v in config.items():
        logger.info(f"  {k}: {v}")
    logger.info("=" * 60)


    logger.info("加载数据集...")
    dataset_wrapper = ContrastiveLearningDataset(
        root_folder=config["data_folder"],
        split_file=config["split_file"]
    )

    train_dataset = dataset_wrapper.get_dataset(config["train_dataset_name"], n_views=2)
    test_dataset  = dataset_wrapper.get_dataset(config["test_dataset_name"],  n_views=2)

    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=config["num_workers"],
        drop_last=True,
        pin_memory=True
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=config["num_workers"],
        pin_memory=True
    )
    logger.info(f"训练集: {len(train_dataset)} 样本 | 测试集: {len(test_dataset)} 样本")


    logger.info("构建模型...")
    model = SpatioTemporalDynamicClassifier(
        num_classes=config["num_classes"],
        d_model=config["d_model"],
        d_state=config["d_state"],
        dropout=config["dropout"],
        hierarchical=config["hierarchical"],
    ).to(device)

    if config["pretrained_path"] and os.path.exists(config["pretrained_path"]):
        load_mambavision_pretrained(model, config["pretrained_path"], verbose=True)
    else:
        logger.info("未找到预训练权重，使用随机初始化")

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(f"可训练参数量: {total_params / 1e6:.2f}M")


    optimizer = build_optimizer(
        model,
        config["lr_backbone"],
        config["lr_head"],
        config["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config["epochs"], eta_min=1e-7
    )
    criterion = nn.CrossEntropyLoss(
        label_smoothing=config["label_smoothing"]
    ).to(device)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


    best_acc = 0.0
    logger.info("开始训练...")

    for epoch in range(config["epochs"]):
        model.train()
        clear_memory()
        total_loss, total_correct, total_count = 0.0, 0, 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['epochs']}")
        for batch in pbar:

            rgb   = batch[0].to(device, non_blocking=True).unsqueeze(1)
            tac   = batch[2].to(device, non_blocking=True).unsqueeze(1)
            label = batch[4].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                logits = model(rgb, tac)
                loss   = criterion(logits, label)

            scaler.scale(loss).backward()

            if config["grad_clip"] > 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])

            scaler.step(optimizer)
            scaler.update()

            pred = logits.argmax(dim=1)
            total_correct += (pred == label).sum().item()
            total_count   += label.size(0)
            total_loss    += loss.item() * label.size(0)
            pbar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "acc":  f"{100.0 * (pred == label).float().mean().item():.2f}%"
            })

        scheduler.step()
        train_acc  = 100.0 * total_correct / max(1, total_count)
        train_loss = total_loss / max(1, total_count)

        test_loss, test_acc = evaluate(
            model, test_loader, device, criterion, use_amp=True
        )

        logger.info(
            f"[Epoch {epoch+1}/{config['epochs']}] "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
            f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}% | "
            f"Best: {best_acc:.2f}%"
        )

        if test_acc > best_acc:
            best_acc = test_acc
            save_path = os.path.join(
                config["save_dir"],
                f"best_objectfolder_{config['train_dataset_name']}_h{config['hierarchical']}.pth"
            )
            torch.save(model.state_dict(), save_path)
            logger.info(f"新最佳模型已保存至 {save_path}")

    logger.info(f"训练完成！最佳测试准确率: {best_acc:.2f}%")


if __name__ == "__main__":
    main()
